# Baseline

These tables contain multiple records for a single client.
Before merging with `application_train` or `application_test`, it is necessary
to engineer features and aggregate the tables to a single row per `SK_ID_CURR`.

Standard numerical features are aggregated in the same way (median, max).
Categorical features are transformed into separate COUNT and SHARE features.
Special features are handled manually based on their meaning.

## Import all

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import gc

from pathlib import Path

from narwhals import Categorical
from narwhals.selectors import categorical
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, precision_score, recall_score, ConfusionMatrixDisplay, confusion_matrix, roc_curve
from lightgbm.callback import early_stopping, log_evaluation

import lightgbm as lgb

In [2]:
path_to_data = "/home/usl/PycharmProjects/home-credit-default-risk/data/raw/home-credit-default-risk/"
# path_to_data = "/content/drive/MyDrive/Home_Credit_data/"

application_train_df = pd.read_csv(path_to_data+"application_train.csv")
application_test_df = pd.read_csv(path_to_data+"application_test.csv")
# bureau_df = pd.read_csv(path_to_data+"bureau.csv")
# bureau_balance_df = pd.read_csv(path_to_data+"bureau_balance.csv")
# credit_card_balance_df = pd.read_csv(path_to_data+"credit_card_balance.csv")
home_credit_columns_description_df = pd.read_csv(path_to_data+"HomeCredit_columns_description.csv", encoding="cp1252")
# installments_payments_df = pd.read_csv(path_to_data+"installments_payments.csv")
# POS_CASH_balance_df = pd.read_csv(path_to_data+"POS_CASH_balance.csv")
# previous_application_df = pd.read_csv(path_to_data+"previous_application.csv")

# Feature engineering

## Main function for numerical and categorical features

### Numerical features: main function

In [3]:
agg_func = ["median", "max"]
def agg_numerical_features(df, id_col, numerical_cols, prefix):
    result = (df.groupby(id_col)[numerical_cols].agg(agg_func))
    result.columns = [prefix + "_" + col + "_" + func for col, func in result.columns]
    return result.astype("float32")

In [4]:
def agg_window_numerical_features(df, id_col, numerical_cols, prefix, sort_col, window_sizes=(3, 5, 10), agg_funcs=("median", "max")):
    sorted_df = df.sort_values(by=[id_col, sort_col])
    result_blocks = []

    for window_size in window_sizes:
        last_n_rows = (sorted_df.groupby(id_col, sort=False).tail(window_size))
        window_features = (last_n_rows.groupby(id_col, sort=False)[numerical_cols].agg(agg_funcs))
        window_features.columns = [f"{prefix}_LAST_{window_size}_{column}_{agg}" for column, agg in window_features.columns]
        record_count = (last_n_rows.groupby(id_col, sort=False).size().rename(f"{prefix}_LAST_{window_size}_RECORD_COUNT"))
        window_features = pd.concat([window_features, record_count], axis=1)
        result_blocks.append(window_features)
    result = pd.concat(result_blocks, axis=1).astype("float32")
    return result

In [5]:
def agg_time_window_numerical_features(df, id_col, time_col, numerical_cols, prefix, window_sizes=(3, 6, 12), unit="M", agg_funcs=("mean", "median", "max")):
    result_blocks = []
    for window_size in window_sizes:

        first_period = -(window_size - 1)
        mask = ((df[time_col] >= first_period) & (df[time_col] <= 0))
        window_df = df.loc[mask, [id_col] + numerical_cols]
        window_features = (window_df.groupby(id_col, sort=False)[numerical_cols].agg(agg_funcs))
        window_features.columns = [f"{prefix}_LAST_{window_size}{unit}_{column}_{agg}" for column, agg in window_features.columns]
        record_count = (window_df.groupby(id_col, sort=False).size().rename(f"{prefix}_LAST_{window_size}{unit}_RECORD_COUNT"))
        window_features = pd.concat([window_features, record_count], axis=1)
        window_features = window_features.astype("float32")
        result_blocks.append(window_features)
    result = pd.concat(result_blocks, axis=1)

    return result

For example:

In [6]:
#numerical_cols_bureau_df = ["AMT_CREDIT_MAX_OVERDUE", "AMT_CREDIT_SUM", "AMT_CREDIT_SUM_DEBT","AMT_CREDIT_SUM_LIMIT", "AMT_CREDIT_SUM_OVERDUE", "AMT_ANNUITY"]

In [7]:
#bureau_num = make_numerical_features(bureau_df, "SK_ID_CURR", numerical_cols_bureau_df, "BUREAU")
#bureau_num

## Categorical features: main function

In [8]:
def agg_categorical_features(df, id_col, categorical_cols, prefix):
        temp = df[[id_col] + categorical_cols].copy()
        temp[categorical_cols] = (temp[categorical_cols].fillna("Missing"))
        dummies = pd.get_dummies(temp[categorical_cols], prefix = [prefix + "_" + col for col in categorical_cols], dtype = np.uint8)
        dummies[id_col] = temp[id_col]
        counts = (dummies.groupby(id_col).sum().astype("float32"))
        shares = (dummies.groupby(id_col).mean().astype("float32"))
        counts.columns = [col + "_COUNT" for col in counts.columns]
        shares.columns = [col + "_SHARE" for col in shares.columns]
        result = pd.concat([counts, shares], axis=1)

        return result

For example:

In [9]:
#categorical_cols_bureau_df = ["CREDIT_ACTIVE", "CREDIT_CURRENCY", "CREDIT_TYPE"]

In [10]:
#bureau_cat = make_categorical_features(bureau_df, "SK_ID_CURR", categorical_cols_bureau_df, "BUREAU")
#bureau_cat

## Main function for tables to client:

In [11]:
def add_contract_feat_to_client(df, prefix):
    sum_cols = [col for col in df.columns if col.endswith("_COUNT") or col.endswith("_SUM")]
    other_cols = [col for col in df.columns if col not in ["SK_ID_PREV", "SK_ID_CURR"] and col not in sum_cols]
    tables = []

    if len(sum_cols) > 0:
        client_sum = (df.groupby("SK_ID_CURR")[sum_cols].sum())
        tables.append(client_sum)

    if len(other_cols) > 0:
        client_other = (df.groupby("SK_ID_CURR")[other_cols].agg(["median", "max"]))

        client_other.columns = [col + "_" + func for col, func in client_other.columns]
        tables.append(client_other)

    contract_count = (df.groupby("SK_ID_CURR")["SK_ID_PREV"].nunique().rename(prefix + "_CONTRACT_WITH_HISTORY_COUNT").to_frame())
    tables.append(contract_count)
    result = pd.concat(tables, axis=1)
    return result

## Bureau_df

In [12]:
bureau_df = pd.read_csv(path_to_data + "bureau.csv")

In [13]:
bureau_df.info(show_counts=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1716428 entries, 0 to 1716427
Data columns (total 17 columns):
 #   Column                  Non-Null Count    Dtype  
---  ------                  --------------    -----  
 0   SK_ID_CURR              1716428 non-null  int64  
 1   SK_ID_BUREAU            1716428 non-null  int64  
 2   CREDIT_ACTIVE           1716428 non-null  object 
 3   CREDIT_CURRENCY         1716428 non-null  object 
 4   DAYS_CREDIT             1716428 non-null  int64  
 5   CREDIT_DAY_OVERDUE      1716428 non-null  int64  
 6   DAYS_CREDIT_ENDDATE     1610875 non-null  float64
 7   DAYS_ENDDATE_FACT       1082775 non-null  float64
 8   AMT_CREDIT_MAX_OVERDUE  591940 non-null   float64
 9   CNT_CREDIT_PROLONG      1716428 non-null  int64  
 10  AMT_CREDIT_SUM          1716415 non-null  float64
 11  AMT_CREDIT_SUM_DEBT     1458759 non-null  float64
 12  AMT_CREDIT_SUM_LIMIT    1124648 non-null  float64
 13  AMT_CREDIT_SUM_OVERDUE  1716428 non-null  float64
 14  CR

In [14]:
bureau_description = home_credit_columns_description_df[home_credit_columns_description_df["Table"].astype(str).str.contains("bureau.csv", case = False, na=False)]
display(bureau_description[["Row", "Description"]])

,Row,Description
122,SK_ID_CURR,ID of loan in our sample - one loan in our sam...
123,SK_BUREAU_ID,Recoded ID of previous Credit Bureau credit re...
124,CREDIT_ACTIVE,Status of the Credit Bureau (CB) reported credits
125,CREDIT_CURRENCY,Recoded currency of the Credit Bureau credit
126,DAYS_CREDIT,How many days before current application did c...
127,CREDIT_DAY_OVERDUE,Number of days past due on CB credit at the ti...
128,DAYS_CREDIT_ENDDATE,Remaining duration of CB credit (in days) at t...
129,DAYS_ENDDATE_FACT,Days since CB credit ended at the time of appl...
130,AMT_CREDIT_MAX_OVERDUE,Maximal amount overdue on the Credit Bureau cr...
131,CNT_CREDIT_PROLONG,How many times was the Credit Bureau credit pr...


Wi can split all features into numerical, categorical and special ones.

In [15]:
categorical_cols_bureau_df = ["CREDIT_ACTIVE", "CREDIT_CURRENCY", "CREDIT_TYPE"]
numerical_cols_bureau_df = ["AMT_CREDIT_MAX_OVERDUE", "AMT_CREDIT_SUM", "AMT_CREDIT_SUM_DEBT","AMT_CREDIT_SUM_LIMIT", "AMT_CREDIT_SUM_OVERDUE", "AMT_ANNUITY"]
special_cols_bureau_df = ["SK_ID_CURR", "SK_ID_BUREAU", "DAYS_CREDIT", "DAYS_CREDIT_ENDDATE","DAYS_ENDDATE_FACT", "DAYS_CREDIT_UPDATE", "CREDIT_DAY_OVERDUE", "CNT_CREDIT_PROLONG"]

In [16]:
print("SPECIAL:", special_cols_bureau_df)
print("\nCATEGORICAL:", categorical_cols_bureau_df)
print("\nNUMERICAL:", numerical_cols_bureau_df)

SPECIAL: ['SK_ID_CURR', 'SK_ID_BUREAU', 'DAYS_CREDIT', 'DAYS_CREDIT_ENDDATE', 'DAYS_ENDDATE_FACT', 'DAYS_CREDIT_UPDATE', 'CREDIT_DAY_OVERDUE', 'CNT_CREDIT_PROLONG']

CATEGORICAL: ['CREDIT_ACTIVE', 'CREDIT_CURRENCY', 'CREDIT_TYPE']

NUMERICAL: ['AMT_CREDIT_MAX_OVERDUE', 'AMT_CREDIT_SUM', 'AMT_CREDIT_SUM_DEBT', 'AMT_CREDIT_SUM_LIMIT', 'AMT_CREDIT_SUM_OVERDUE', 'AMT_ANNUITY']


### Numerical

In [17]:
bureau_num = agg_numerical_features(bureau_df, "SK_ID_CURR", numerical_cols_bureau_df, "BUREAU")
bureau_num

,BUREAU_AMT_CREDIT_MAX_OVERDUE_median,BUREAU_AMT_CREDIT_MAX_OVERDUE_max,BUREAU_AMT_CREDIT_SUM_median,BUREAU_AMT_CREDIT_SUM_max,BUREAU_AMT_CREDIT_SUM_DEBT_median,BUREAU_AMT_CREDIT_SUM_DEBT_max,BUREAU_AMT_CREDIT_SUM_LIMIT_median,BUREAU_AMT_CREDIT_SUM_LIMIT_max,BUREAU_AMT_CREDIT_SUM_OVERDUE_median,BUREAU_AMT_CREDIT_SUM_OVERDUE_max,BUREAU_AMT_ANNUITY_median,BUREAU_AMT_ANNUITY_max
SK_ID_CURR,,,,,,,,,,,,
100001,NaN,NaN,168345.000000,3.780000e+05,0.000000,373239.0,0.0,0.000000,0.0,0.0,0.0,10822.5
100002,40.500000,5043.64502,54130.500000,4.500000e+05,0.000000,245781.0,0.0,31988.564453,0.0,0.0,0.0,0.0
100003,0.000000,0.00000,92576.250000,8.100000e+05,0.000000,0.0,0.0,810000.000000,0.0,0.0,NaN,NaN
100004,0.000000,0.00000,94518.898438,9.453780e+04,0.000000,0.0,0.0,0.000000,0.0,0.0,NaN,NaN
100005,0.000000,0.00000,58500.000000,5.688000e+05,25321.500000,543087.0,0.0,0.000000,0.0,0.0,0.0,4261.5
...,...,...,...,...,...,...,...,...,...,...,...,...
456249,0.000000,18945.00000,248692.500000,7.650000e+05,0.000000,163071.0,0.0,0.000000,0.0,0.0,NaN,NaN
456250,0.000000,0.00000,483349.500000,2.153110e+06,391731.625000,1840308.5,0.0,58268.386719,0.0,0.0,51799.5,384147.0
456253,NaN,NaN,675000.000000,2.250000e+06,85518.000000,1624797.0,0.0,0.000000,0.0,0.0,58369.5,58369.5


In [18]:
bureau_df["DEBT_TO_CREDIT"] = (bureau_df["AMT_CREDIT_SUM_DEBT"] / bureau_df["AMT_CREDIT_SUM"].replace(0, np.nan))
bureau_df["OVERDUE_TO_CREDIT"] = (bureau_df["AMT_CREDIT_SUM_OVERDUE"] / bureau_df["AMT_CREDIT_SUM"].replace(0, np.nan))

In [19]:
bureau_window_cols = (numerical_cols_bureau_df + ["CREDIT_DAY_OVERDUE", "CNT_CREDIT_PROLONG", "DEBT_TO_CREDIT", "OVERDUE_TO_CREDIT"])
bureau_num_window = agg_window_numerical_features(
    df=bureau_df,
    id_col="SK_ID_CURR",
    numerical_cols=bureau_window_cols,
    prefix="BUREAU",
    sort_col="DAYS_CREDIT",
    window_sizes=(3, 5, 10)
)
bureau_num_window.head()

,BUREAU_LAST_3_AMT_CREDIT_MAX_OVERDUE_median,BUREAU_LAST_3_AMT_CREDIT_MAX_OVERDUE_max,BUREAU_LAST_3_AMT_CREDIT_SUM_median,BUREAU_LAST_3_AMT_CREDIT_SUM_max,BUREAU_LAST_3_AMT_CREDIT_SUM_DEBT_median,BUREAU_LAST_3_AMT_CREDIT_SUM_DEBT_max,BUREAU_LAST_3_AMT_CREDIT_SUM_LIMIT_median,BUREAU_LAST_3_AMT_CREDIT_SUM_LIMIT_max,BUREAU_LAST_3_AMT_CREDIT_SUM_OVERDUE_median,BUREAU_LAST_3_AMT_CREDIT_SUM_OVERDUE_max,...,BUREAU_LAST_10_AMT_ANNUITY_max,BUREAU_LAST_10_CREDIT_DAY_OVERDUE_median,BUREAU_LAST_10_CREDIT_DAY_OVERDUE_max,BUREAU_LAST_10_CNT_CREDIT_PROLONG_median,BUREAU_LAST_10_CNT_CREDIT_PROLONG_max,BUREAU_LAST_10_DEBT_TO_CREDIT_median,BUREAU_LAST_10_DEBT_TO_CREDIT_max,BUREAU_LAST_10_OVERDUE_TO_CREDIT_median,BUREAU_LAST_10_OVERDUE_TO_CREDIT_max,BUREAU_LAST_10_RECORD_COUNT
SK_ID_CURR,,,,,,,,,,,,,,,,,,,,,
100001,NaN,NaN,337680.000000,378000.000000,113166.0,373239.0,0.000000,0.000000,0.0,0.0,...,10822.5,0.0,0.0,0.0,0.0,0.000000,0.987405,0.0,0.0,7.0
100002,2542.07251,5043.64502,31988.564453,120735.000000,0.0,0.0,15994.282227,31988.564453,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.546180,0.0,0.0,8.0
100003,0.00000,0.00000,112500.000000,810000.000000,0.0,0.0,0.000000,810000.000000,0.0,0.0,...,NaN,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,4.0
100004,0.00000,0.00000,94518.898438,94537.796875,0.0,0.0,0.000000,0.000000,0.0,0.0,...,NaN,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,2.0
100005,0.00000,0.00000,58500.000000,568800.000000,25321.5,543087.0,0.000000,0.000000,0.0,0.0,...,4261.5,0.0,0.0,0.0,0.0,0.848974,0.954794,0.0,0.0,3.0


In [ ]:
bureau_last = (bureau_df.sort_values(["SK_ID_CURR", "DAYS_CREDIT"]).groupby("SK_ID_CURR").tail(1).set_index("SK_ID_CURR")[numerical_cols_bureau_df + ["DAYS_CREDIT"]].copy())
bureau_last.columns = ["BUREAU_LAST_" + col for col in bureau_last.columns]

### Trend

In [ ]:
bureau_trend = pd.DataFrame(index=bureau_num_window.index)

In [ ]:
for col in numerical_cols_bureau_df:
    bureau_trend["BUREAU_TREND_3_10_" + col] = (bureau_num_window["BUREAU_LAST_3_" + col + "_median"] - bureau_num_window["BUREAU_LAST_10_" + col + "_median"])

In [ ]:
for col in numerical_cols_bureau_df:
    bureau_trend["BUREAU_TREND_5_10_" + col] = (bureau_num_window["BUREAU_LAST_5_" + col + "_median"] - bureau_num_window["BUREAU_LAST_10_" + col + "_median"])

### Categorical

In [ ]:
bureau_cat = agg_categorical_features(bureau_df, "SK_ID_CURR", categorical_cols_bureau_df, "BUREAU")
bureau_cat

### Special

We can calculate the count of credits:

In [ ]:
bureau_credit_count = (bureau_df.groupby("SK_ID_CURR")["SK_ID_BUREAU"].count().rename("BUREAU_CREDIT_COUNT"))
bureau_credit_count.shape

In [ ]:
#bureau_df["CREDIT_ACTIVE"].value_counts(dropna=False)

And we can check whether there is active credit or not:

In [ ]:
bureau_df["IS_ACTIVE_CREDIT"] = (bureau_df["CREDIT_ACTIVE"] == "Active").astype(int)
bureau_active = (bureau_df.groupby("SK_ID_CURR").agg(BUREAU_ACTIVE_COUNT = ("IS_ACTIVE_CREDIT", "sum"), BUREAU_ACTIVE_SHARE = ("IS_ACTIVE_CREDIT", "mean")))
bureau_active.head()

We can check how many days of credit are available:

In [ ]:
bureau_df["DAYS_CREDIT"].describe()


max indicates how recently the last loan was taken out,
min indicates how far back the credit history goes,
median typical age of the client's loans

In [ ]:
bureau_days_credit = (bureau_df.groupby("SK_ID_CURR").agg(BUREAU_MOST_RECENT_CREDIT_DAYS = ("DAYS_CREDIT", "max"), BUREAU_OLDEST_CREDIT_DAYS = ("DAYS_CREDIT", "min"), BUREAU_MEDIAN_CREDIT_DAYS = ("DAYS_CREDIT", "median")))
bureau_days_credit.head()

We can obtain the length of the client's credit history:

In [ ]:
bureau_days_credit["BUREAU_CREDIT_HISTORY_LENGTH"] = (bureau_days_credit["BUREAU_MOST_RECENT_CREDIT_DAYS"] - bureau_days_credit["BUREAU_OLDEST_CREDIT_DAYS"])
bureau_days_credit

In [ ]:
#bureau_df["CREDIT_DAY_OVERDUE"].describe()

On this part we look at overdue:

In [ ]:
bureau_df["IS_DAY_OVERDUE"] = (bureau_df["CREDIT_DAY_OVERDUE"] > 0).astype(int)

In [ ]:
bureau_overdue = (bureau_df.groupby("SK_ID_CURR").agg(BUREAU_MAX_DAYS_OVERDUE=("CREDIT_DAY_OVERDUE", "max"), BUREAU_OVERDUE_CREDIT_COUNT=("IS_DAY_OVERDUE", "sum"), BUREAU_OVERDUE_CREDIT_SHARE=("IS_DAY_OVERDUE", "mean")))
bureau_overdue.head(20)

In [ ]:
# print("Доля записей с просрочкой:", (bureau_df["CREDIT_DAY_OVERDUE"] > 0).mean())
# print("Количество записей с просрочкой:", (bureau_df["CREDIT_DAY_OVERDUE"] > 0).sum())

In [ ]:
#(bureau_overdue["BUREAU_OVERDUE_CREDIT_COUNT"] > 0).mean()

In [ ]:
# temp = application_train_df[["SK_ID_CURR", "TARGET"]].merge(bureau_overdue, on="SK_ID_CURR", how="left")
# temp["HAS_BUREAU_OVERDUE"] = (temp["BUREAU_OVERDUE_CREDIT_COUNT"].fillna(0) > 0).astype(int)
# temp.groupby("HAS_BUREAU_OVERDUE")["TARGET"].agg(["count", "mean"])

We will collect data on the loan extension:

In [ ]:
bureau_df["IS_PROLONGED"] = (bureau_df["CNT_CREDIT_PROLONG"] > 0).astype(int)
bureau_prolong = bureau_df.groupby("SK_ID_CURR").agg(BUREAU_TOTAL_PROLONG = ("CNT_CREDIT_PROLONG", "sum"), BUREAU_MAX_PROLONG = ("CNT_CREDIT_PROLONG", "max"), BUREAU_PROLONG_SHARE = ("IS_PROLONGED", "mean"))
bureau_prolong

We will collect data of the days credit end data:

In [ ]:
bureau_df["ENDS_AFTER_APPLICATION"] = np.where(bureau_df["DAYS_CREDIT_ENDDATE"].isna(), np.nan, (bureau_df["DAYS_CREDIT_ENDDATE"] > 0).astype(int))
bureau_credit_enddate = bureau_df.groupby("SK_ID_CURR").agg(BUREAU_LAST_PLANNED_ENDDATE=("DAYS_CREDIT_ENDDATE", "max"), BUREAU_MEAN_PLANNED_ENDDATE=("DAYS_CREDIT_ENDDATE", "mean"), BUREAU_ENDS_AFTER_APPL_SHARE=("ENDS_AFTER_APPLICATION", "mean"))
bureau_credit_enddate

In [ ]:
bureau_fact_enddate = (bureau_df.groupby("SK_ID_CURR").agg(BUREAU_LAST_FACT_ENDDATE = ("DAYS_ENDDATE_FACT", "max"), BUREAU_MEAN_FACT_ENDDATE = ("DAYS_ENDDATE_FACT", "mean")))
bureau_fact_enddate

And we will find the difference between the planned (fact) and actual dates:

In [ ]:
bureau_df["ENDDATE_DIFF"] = bureau_df["DAYS_ENDDATE_FACT"] - bureau_df["DAYS_CREDIT_ENDDATE"]
bureau_enddate_diff = (bureau_df.groupby("SK_ID_CURR").agg(BUREAU_MEDIAN_DIFF_ENDDATE = ("ENDDATE_DIFF", "median"), BUREAU_MAX_DIFF_ENDDATE = ("ENDDATE_DIFF", "max")))
bureau_enddate_diff

In [ ]:
bureau_update = (bureau_df.groupby("SK_ID_CURR").agg(BUREAU_LAST_UPDATE_DAYS = ("DAYS_CREDIT_UPDATE", "max"), BUREAU_MEAN_UPDATE_DAYS = ("DAYS_CREDIT_UPDATE", "mean")))
bureau_update

We now synthesize the ratio of monetary quantities to others:

In [ ]:
bureau_ratios = bureau_df.groupby("SK_ID_CURR").agg(BUREAU_MEDIAN_DEBT_RATIO = ("DEBT_TO_CREDIT", "median"), BUREAU_MAX_DEBT_RATIO = ("DEBT_TO_CREDIT", "max"), BUREAU_MEDIAN_OVERDUE_RATIO = ("OVERDUE_TO_CREDIT", "median"), BUREAU_MAX_OVERDUE_RATIO = ("OVERDUE_TO_CREDIT", "max"))
bureau_ratios

Finally, we will gather all the special features:

In [ ]:
bureau_special = pd.concat(
    [
        bureau_credit_count,
        bureau_active,
        bureau_days_credit,
        bureau_overdue,
        bureau_prolong,
        bureau_credit_enddate,
        bureau_fact_enddate,
        bureau_enddate_diff,
        bureau_update,
        bureau_ratios
    ],
    axis=1
)
bureau_special.head()

All features for bureau_df:

In [ ]:
bureau_features = pd.concat([
    bureau_num,
    bureau_num_window,
    bureau_last,
    # bureau_trend,
    bureau_cat,
    bureau_special
], axis=1)
bureau_features.head()

In [ ]:
# print("Размер:", bureau_features.shape)
# print("Уникальный SK_ID_CURR:", bureau_features.index.is_unique)
# print("Количество клиентов:", bureau_features.shape[0])

## Bureau_balance_df

In [ ]:
bureau_balance_df = pd.read_csv(path_to_data+"bureau_balance.csv")

In [ ]:
bureau_balance_df.info(show_counts=True)
bureau_balance_df.shape

In [ ]:
home_credit_columns_description_df = pd.read_csv(path_to_data+"HomeCredit_columns_description.csv", encoding="cp1252")
bureau_balance_description = home_credit_columns_description_df[home_credit_columns_description_df["Table"].astype(str).str.contains("bureau_balance.csv", case = False, na=False)]
display(bureau_balance_description[["Row", "Description"]])

In [ ]:
categorical_cols_bureau_balance_df = ["STATUS"]
numerical_cols_bureau_balance_df = []
special_cols_bureau_balance_df = ["SK_ID_BUREAU", "MONTHS_BALANCE"]

In [ ]:
print("SPECIAL:", special_cols_bureau_balance_df)
print("\nCATEGORICAL:", categorical_cols_bureau_balance_df)
print("\nNUMERICAL:", numerical_cols_bureau_balance_df)

### Categorical

In [ ]:
bureau_balance_df["STATUS"].value_counts(dropna=False)

In [ ]:
bureau_balance_df["IS_BB_OVERDUE"] = bureau_balance_df["STATUS"].isin(["1", "2", "3", "4", "5"]).astype(int)

In [ ]:
bureau_balance_df["IS_BB_OVERDUE"].value_counts()

In [ ]:
bureau_balance_df["IS_BB_OVERDUE"].mean()

We can add hard overdue:

In [ ]:
bureau_balance_df["IS_BB_SEVERE_OVERDUE"] = bureau_balance_df["STATUS"].isin(["3", "4", "5"]).astype(int)

In [ ]:
bureau_to_client = (bureau_df.drop_duplicates("SK_ID_BUREAU").set_index("SK_ID_BUREAU")["SK_ID_CURR"])
bureau_balance_df["SK_ID_CURR"] = (bureau_balance_df["SK_ID_BUREAU"].map(bureau_to_client))

In [ ]:
status_to_number = {
    "0": 0,
    "1": 1,
    "2": 2,
    "3": 3,
    "4": 4,
    "5": 5,
    "C": 0,
    "X": np.nan
}

bureau_balance_df["BB_STATUS_LEVEL"] = (bureau_balance_df["STATUS"].map(status_to_number).astype("float32"))

In [ ]:
bb_window_features = agg_time_window_numerical_features(
    df=bureau_balance_df,
    id_col="SK_ID_CURR",
    time_col="MONTHS_BALANCE",
    numerical_cols=["IS_BB_OVERDUE", "IS_BB_SEVERE_OVERDUE", "BB_STATUS_LEVEL"],
    prefix="BB",
    window_sizes=(3, 6, 12),
    unit="M"
)

bb_window_features.head()

In [ ]:
bureau_balance_cat = agg_categorical_features(bureau_balance_df, "SK_ID_BUREAU", categorical_cols_bureau_balance_df, "BB")
bureau_balance_cat

### Special

In [ ]:
bureau_balance_df["MONTHS_BALANCE"].value_counts(dropna=False)

In [ ]:
bureau_balance_special = (bureau_balance_df.groupby("SK_ID_BUREAU").agg(BB_MONTH_COUNT=("MONTHS_BALANCE","count"),
BB_OLDEST_MONTH=("MONTHS_BALANCE","min"), BB_MOST_RECENT_MONTH=("MONTHS_BALANCE", "max"), BB_OVERDUE_MONTH_COUNT=("IS_BB_OVERDUE", "sum"), BB_SEVERE_OVERDUE_MONTH_COUNT=("IS_BB_SEVERE_OVERDUE", "sum")))
bureau_balance_special

In [ ]:
bureau_balance_special["BB_HISTORY_LENGTH"] = (bureau_balance_special["BB_MOST_RECENT_MONTH"] - bureau_balance_special["BB_OLDEST_MONTH"])

In [ ]:
bureau_balance_features = pd.concat([bureau_balance_cat, bureau_balance_special], axis=1)
bureau_balance_features.head()

In [ ]:
# print("Уникальный SK_ID_BUREAU:", bureau_balance_features.index.is_unique)
# print("Размер:", bureau_balance_features.shape)

We will merge 3 tables: bureau_df and bureau_balance_df to application_train_df

In [ ]:
bureau_balance_with_client = (bureau_balance_features.reset_index().merge(bureau_df[["SK_ID_BUREAU", "SK_ID_CURR"]], on = "SK_ID_BUREAU", how = "left", validate = "1:1"))
bureau_balance_with_client.head()

In [ ]:
# print("Кредитов без SK_ID_CURR:", bureau_balance_with_client["SK_ID_CURR"].isna().sum())

In [ ]:
bb_ids = set(bureau_balance_features.index)
bureau_ids = set(bureau_df["SK_ID_BUREAU"])
unmapped_ids = (bb_ids - bureau_ids)

print("SK_ID_BUREAU в bureau_balance:", len(bb_ids))
print("SK_ID_BUREAU в bureau:", len(bureau_ids))
print("Есть в bureau_balance, но нет в bureau:", len(unmapped_ids))

In [ ]:
bureau_balance_mapped = (bureau_balance_with_client[bureau_balance_with_client["SK_ID_CURR"].notna()].copy())
bureau_balance_unmapped = (bureau_balance_with_client[bureau_balance_with_client["SK_ID_CURR"].isna()].copy())

In [ ]:
bureau_balance_mapped["SK_ID_CURR"] = (bureau_balance_mapped["SK_ID_CURR"].astype("int64"))

Use "count" for 1 client:

In [ ]:
bureau_balance_count_cols = [col for col in bureau_balance_mapped.columns if col.endswith("_COUNT")]
bureau_balance_count_cols

In [ ]:
bureau_balance_client_counts = (bureau_balance_mapped.groupby("SK_ID_CURR")[bureau_balance_count_cols].sum())
bureau_balance_client_counts.head()

We will add the remaining customer attributes:

In [ ]:
bureau_balance_client_special = (bureau_balance_mapped.groupby("SK_ID_CURR").agg(BB_CREDIT_WITH_HISTORY_COUNT = ("SK_ID_BUREAU", "nunique"), BB_HISTORY_LENGTH_MAX = ("BB_HISTORY_LENGTH", "max"), BB_HISTORY_LENGTH_MEDIAN = ("BB_HISTORY_LENGTH", "median"), BB_OLDEST_MONTH_CLIENT = ("BB_OLDEST_MONTH", "min"), BB_MOST_RECENT_MONTH_CLIENT = ("BB_MOST_RECENT_MONTH", "max")))
bureau_balance_client_special

We calculate the shares of overdue payments for each client:

In [ ]:
bureau_balance_client_counts["BB_OVERDUE_MONTH_SHARE"] = (bureau_balance_client_counts["BB_OVERDUE_MONTH_COUNT"] / bureau_balance_client_counts["BB_MONTH_COUNT"].replace(0, np.nan))

bureau_balance_client_counts["BB_SEVERE_OVERDUE_MONTH_SHARE"] = (bureau_balance_client_counts["BB_SEVERE_OVERDUE_MONTH_COUNT"] / bureau_balance_client_counts["BB_MONTH_COUNT"].replace(0, np.nan))

In [ ]:
bureau_balance_status_count_cols = [col for col in bureau_balance_client_counts.columns if col.startswith("BB_STATUS_")and col.endswith("_COUNT")]

In [ ]:
for col in bureau_balance_status_count_cols:

    share_col = col.replace("_COUNT", "_SHARE")
    bureau_balance_client_counts[share_col] = (bureau_balance_client_counts[col] / bureau_balance_client_counts["BB_MONTH_COUNT"].replace(0, np.nan))

Add final tables:

In [ ]:
bureau_balance_features_client = pd.concat([bureau_balance_client_counts, bureau_balance_client_special, bb_window_features], axis=1)
bureau_balance_features_client.head()

In [ ]:
# print("Размер:", bureau_balance_features_client.shape)
# print("SK_ID_CURR уникален:", bureau_balance_features_client.index.is_unique)
# print("Пропусков в ID:", bureau_balance_features_client.index.isna().sum())

In [ ]:
del bureau_balance_df

del bureau_balance_features
del bureau_balance_with_client
del bureau_balance_mapped
del bureau_balance_unmapped

del bureau_balance_cat
del bureau_balance_special

del bureau_balance_client_counts
del bureau_balance_client_special

del bureau_df

del bureau_num
del bureau_cat
del bureau_special

del bureau_num_window
del bureau_last
# del bureau_trend

del bb_window_features
del bureau_to_client

del bb_ids
del bureau_ids
del unmapped_ids

gc.collect()

In [ ]:
bureau_features = bureau_features.astype("float32")
bureau_balance_features_client = (bureau_balance_features_client.astype("float32"))

gc.collect()

## Previous_application_df

In [ ]:
previous_application_df = pd.read_csv(path_to_data+"previous_application.csv")

In [ ]:
previous_application_df.info(show_counts=True)
previous_application_df.shape

In [ ]:
home_credit_columns_description_df = pd.read_csv(path_to_data+"HomeCredit_columns_description.csv", encoding="cp1252")
previous_application_description = home_credit_columns_description_df[home_credit_columns_description_df["Table"].astype(str).str.contains("previous_application.csv", case = False, na=False)]
display(previous_application_description[["Row", "Description"]])

In [ ]:
categorical_cols_prev_app_df = ["NAME_CONTRACT_TYPE", "WEEKDAY_APPR_PROCESS_START","FLAG_LAST_APPL_PER_CONTRACT", "NAME_CASH_LOAN_PURPOSE", "NAME_CONTRACT_STATUS","NAME_PAYMENT_TYPE", "CODE_REJECT_REASON", "NAME_TYPE_SUITE", "NAME_CLIENT_TYPE","NAME_GOODS_CATEGORY", "NAME_PORTFOLIO", "NAME_PRODUCT_TYPE", "CHANNEL_TYPE","NAME_SELLER_INDUSTRY", "NAME_YIELD_GROUP", "PRODUCT_COMBINATION"]
numerical_cols_prev_app_df = ["AMT_ANNUITY", "AMT_APPLICATION", "AMT_CREDIT","AMT_DOWN_PAYMENT", "AMT_GOODS_PRICE", "RATE_DOWN_PAYMENT", "RATE_INTEREST_PRIMARY", "RATE_INTEREST_PRIVILEGED"]
special_cols_prev_app_df = ["SK_ID_PREV", "SK_ID_CURR", "HOUR_APPR_PROCESS_START", "NFLAG_LAST_APPL_IN_DAY", "CNT_PAYMENT", "DAYS_DECISION", "DAYS_FIRST_DRAWING", "DAYS_FIRST_DUE", "DAYS_LAST_DUE_1ST_VERSION", "DAYS_LAST_DUE", "DAYS_TERMINATION", "NFLAG_INSURED_ON_APPROVAL", "SELLERPLACE_AREA"]

In [ ]:
print("SPECIAL:", special_cols_prev_app_df)
print("\nCATEGORICAL:", categorical_cols_prev_app_df)
print("\nNUMERICAL:", numerical_cols_prev_app_df)

In [ ]:
prev_work = previous_application_df

del previous_application_df
gc.collect()

In [ ]:
prev_days_cols = ["DAYS_FIRST_DRAWING", "DAYS_FIRST_DUE", "DAYS_LAST_DUE_1ST_VERSION", "DAYS_LAST_DUE", "DAYS_TERMINATION"]
prev_work[prev_days_cols] = (prev_work[prev_days_cols].replace(365243, np.nan))

In [ ]:
prev_work["CREDIT_TO_APPLICATION"] = (prev_work["AMT_CREDIT"] / prev_work["AMT_APPLICATION"].replace(0, np.nan))
prev_work["DOWN_PAYMENT_SHARE"] = (prev_work["AMT_DOWN_PAYMENT"] / prev_work["AMT_APPLICATION"].replace(0, np.nan))
prev_work["ESTIMATED_TOTAL_PAYMENT"] = (prev_work["AMT_ANNUITY"] * prev_work["CNT_PAYMENT"])
prev_work["TOTAL_PAYMENT_TO_CREDIT"] = (prev_work["ESTIMATED_TOTAL_PAYMENT"] / prev_work["AMT_CREDIT"].replace(0, np.nan))
prev_work["PLANNED_DURATION"] = (prev_work["DAYS_LAST_DUE_1ST_VERSION"] - prev_work["DAYS_FIRST_DUE"])
prev_work["ACTUAL_DURATION"] = (prev_work["DAYS_LAST_DUE"] - prev_work["DAYS_FIRST_DUE"])

### Numerical

In [ ]:
prev_num = agg_numerical_features(prev_work, "SK_ID_CURR", numerical_cols_prev_app_df, "PREV")
prev_num.head()

In [ ]:
prev_window_cols = (numerical_cols_prev_app_df + ["CNT_PAYMENT", "CREDIT_TO_APPLICATION", "DOWN_PAYMENT_SHARE", "ESTIMATED_TOTAL_PAYMENT", "TOTAL_PAYMENT_TO_CREDIT", "PLANNED_DURATION", "ACTUAL_DURATION"])

In [ ]:
prev_num_window = agg_window_numerical_features(
    df=prev_work,
    id_col="SK_ID_CURR",
    numerical_cols=prev_window_cols,
    prefix="PREV",
    sort_col="DAYS_DECISION",
    window_sizes=(3, 5, 10)
)
prev_num_window.head()

### Categorical

In [ ]:
prev_cat = agg_categorical_features(prev_work, "SK_ID_CURR", categorical_cols_prev_app_df, "PREV")
prev_cat.head()

### Special

In [ ]:
prev_special = (
    prev_work
    .groupby("SK_ID_CURR")
    .agg(
        PREV_APPLICATION_COUNT=(
            "SK_ID_PREV",
            "nunique"
        ),

        PREV_MOST_RECENT_DECISION=(
            "DAYS_DECISION",
            "max"
        ),

        PREV_OLDEST_DECISION=(
            "DAYS_DECISION",
            "min"
        ),

        PREV_CNT_PAYMENT_MEDIAN=(
            "CNT_PAYMENT",
            "median"
        ),

        PREV_CNT_PAYMENT_MAX=(
            "CNT_PAYMENT",
            "max"
        ),

        PREV_INSURED_SHARE=(
            "NFLAG_INSURED_ON_APPROVAL",
            "mean"
        ),

        PREV_LAST_APPL_IN_DAY_SHARE=(
            "NFLAG_LAST_APPL_IN_DAY",
            "mean"
        ),

        PREV_CREDIT_TO_APPLICATION_MEDIAN=(
            "CREDIT_TO_APPLICATION",
            "median"
        ),

        PREV_CREDIT_TO_APPLICATION_MAX=(
            "CREDIT_TO_APPLICATION",
            "max"
        ),

        PREV_DOWN_PAYMENT_SHARE_MEDIAN=(
            "DOWN_PAYMENT_SHARE",
            "median"
        ),

        PREV_TOTAL_PAYMENT_TO_CREDIT_MEDIAN=(
            "TOTAL_PAYMENT_TO_CREDIT",
            "median"
        )
    )
)

prev_special.head()

In [ ]:
prev_features = pd.concat([prev_num, prev_num_window, prev_cat, prev_special], axis=1)
prev_features.head()

In [ ]:
prev_id_map = (prev_work[["SK_ID_PREV", "SK_ID_CURR"]].copy())

In [ ]:
del prev_work
del prev_num
del prev_cat
del prev_special
del prev_num_window

gc.collect()

In [ ]:
prev_features = prev_features.astype("float32")
gc.collect()

## POS_CASH_balance_df

In [ ]:
POS_CASH_balance_df = pd.read_csv(path_to_data+"POS_CASH_balance.csv")

In [ ]:
POS_CASH_balance_df.info(show_counts=True)
POS_CASH_balance_df.shape

In [ ]:
home_credit_columns_description_df = pd.read_csv(path_to_data+"HomeCredit_columns_description.csv", encoding="cp1252")
POS_CASH_balance_description = home_credit_columns_description_df[home_credit_columns_description_df["Table"].astype(str).str.contains("POS_CASH_balance.csv", case = False, na=False)]
display(POS_CASH_balance_description[["Row", "Description"]])


In [ ]:
categorical_cols_POS_CASH_bal_df = ["NAME_CONTRACT_STATUS"]
numerical_cols_POS_CASH_bal_df = ["CNT_INSTALMENT", "CNT_INSTALMENT_FUTURE"]
special_cols_POS_CASH_bal_df = ["SK_ID_PREV", "SK_ID_CURR", "MONTHS_BALANCE", "SK_DPD", "SK_DPD_DEF"]

In [ ]:
print("SPECIAL:", special_cols_POS_CASH_bal_df)
print("\nCATEGORICAL:", categorical_cols_POS_CASH_bal_df)
print("\nNUMERICAL:", numerical_cols_POS_CASH_bal_df)

### Numerical

In [ ]:
POS_CASH_bal_work = POS_CASH_balance_df
del POS_CASH_balance_df
gc.collect()

In [ ]:
POS_CASH_bal_work["POS_IS_OVERDUE"] = (POS_CASH_bal_work["SK_DPD"] > 0).astype(int)
POS_CASH_bal_work["POS_IS_OVERDUE_DEF"] = (POS_CASH_bal_work["SK_DPD_DEF"] > 0).astype(int)

In [ ]:
POS_CASH_bal_work["POS_COMPLETION_RATIO"] = ((POS_CASH_bal_work["CNT_INSTALMENT"] - POS_CASH_bal_work["CNT_INSTALMENT_FUTURE"]) / POS_CASH_bal_work["CNT_INSTALMENT"].replace(0, np.nan))

In [ ]:
pos_num_prev = agg_numerical_features(POS_CASH_bal_work, "SK_ID_PREV", numerical_cols_POS_CASH_bal_df, "POS")
pos_num_prev.head()

In [ ]:
pos_window_features = agg_time_window_numerical_features(
    df=POS_CASH_bal_work,
    id_col="SK_ID_CURR",
    time_col="MONTHS_BALANCE",
    numerical_cols=["CNT_INSTALMENT", "CNT_INSTALMENT_FUTURE", "SK_DPD", "SK_DPD_DEF", "POS_IS_OVERDUE", "POS_IS_OVERDUE_DEF", "POS_COMPLETION_RATIO"],
    prefix="POS",
    window_sizes=(3, 6, 12),
    unit="M"
)

pos_window_features.head()

### Categorical

In [ ]:
pos_cat_prev = agg_categorical_features(POS_CASH_bal_work, "SK_ID_PREV", categorical_cols_POS_CASH_bal_df, "POS")
pos_cat_prev.head()

### Special

In [ ]:
pos_special_prev = (
    POS_CASH_bal_work
    .groupby("SK_ID_PREV")
    .agg(
        POS_MONTH_COUNT=(
            "MONTHS_BALANCE",
            "count"
        ),

        POS_OLDEST_MONTH=(
            "MONTHS_BALANCE",
            "min"
        ),

        POS_MOST_RECENT_MONTH=(
            "MONTHS_BALANCE",
            "max"
        ),

        POS_MAX_DPD=(
            "SK_DPD",
            "max"
        ),

        POS_MAX_DPD_DEF=(
            "SK_DPD_DEF",
            "max"
        ),

        POS_OVERDUE_MONTH_COUNT=(
            "POS_IS_OVERDUE",
            "sum"
        ),

        POS_OVERDUE_DEF_MONTH_COUNT=(
            "POS_IS_OVERDUE_DEF",
            "sum"
        ),

        POS_COMPLETION_RATIO_MEDIAN=(
            "POS_COMPLETION_RATIO",
            "median"
        ),

        POS_COMPLETION_RATIO_MAX=(
            "POS_COMPLETION_RATIO",
            "max"
        )
    )
)

pos_special_prev.head()

In [ ]:
pos_special_prev["POS_HISTORY_LENGTH"] = (pos_special_prev["POS_MOST_RECENT_MONTH"] - pos_special_prev["POS_OLDEST_MONTH"])

In [ ]:
pos_features_prev = pd.concat([pos_num_prev, pos_cat_prev, pos_special_prev], axis=1)
pos_features_prev.head()

In [ ]:
pos_with_client = (pos_features_prev.reset_index().merge(prev_id_map, on="SK_ID_PREV", how="left", validate="1:1"))
pos_with_client.head()

In [ ]:
# print("POS договоров без SK_ID_CURR:", pos_with_client["SK_ID_CURR"].isna().sum())

In [ ]:
pos_mapped = (pos_with_client[pos_with_client["SK_ID_CURR"].notna()].copy())
pos_unmapped = (pos_with_client[pos_with_client["SK_ID_CURR"].isna()].copy())

In [ ]:
pos_count_cols = [col for col in pos_mapped.columns if col.endswith("_COUNT")]

In [ ]:
pos_client_counts = (pos_mapped.groupby("SK_ID_CURR")[pos_count_cols].sum())
pos_client_counts.head()

In [ ]:
pos_client_counts["POS_OVERDUE_MONTH_SHARE"] = (pos_client_counts["POS_OVERDUE_MONTH_COUNT"] / pos_client_counts["POS_MONTH_COUNT"].replace(0, np.nan))
pos_client_counts["POS_OVERDUE_DEF_MONTH_SHARE"] = (pos_client_counts["POS_OVERDUE_DEF_MONTH_COUNT"] / pos_client_counts["POS_MONTH_COUNT"].replace(0, np.nan))

In [ ]:
pos_status_count_cols = [col for col in pos_client_counts.columns if col.startswith("POS_NAME_CONTRACT_STATUS_") and col.endswith("_COUNT")]

In [ ]:
for col in pos_status_count_cols:

    share_col = col.replace("_COUNT", "_SHARE")
    pos_client_counts[share_col] = (pos_client_counts[col] / pos_client_counts["POS_MONTH_COUNT"].replace(0, np.nan))

In [ ]:
pos_client_special = (
    pos_mapped
    .groupby("SK_ID_CURR")
    .agg(
        POS_CONTRACT_WITH_HISTORY_COUNT=(
            "SK_ID_PREV",
            "nunique"
        ),

        POS_HISTORY_LENGTH_MAX=(
            "POS_HISTORY_LENGTH",
            "max"
        ),

        POS_HISTORY_LENGTH_MEDIAN=(
            "POS_HISTORY_LENGTH",
            "median"
        ),

        POS_OLDEST_MONTH_CLIENT=(
            "POS_OLDEST_MONTH",
            "min"
        ),

        POS_MOST_RECENT_MONTH_CLIENT=(
            "POS_MOST_RECENT_MONTH",
            "max"
        ),

        POS_MAX_DPD_CLIENT=(
            "POS_MAX_DPD",
            "max"
        ),

        POS_MAX_DPD_DEF_CLIENT=(
            "POS_MAX_DPD_DEF",
            "max"
        ),

        POS_COMPLETION_RATIO_MEDIAN_CLIENT=(
            "POS_COMPLETION_RATIO_MEDIAN",
            "median"
        ),

        POS_COMPLETION_RATIO_MAX_CLIENT=(
            "POS_COMPLETION_RATIO_MAX",
            "max"
        )
    )
)

In [ ]:
pos_numeric_prev_cols = list(pos_num_prev.columns)

In [ ]:
pos_client_num = (pos_mapped.groupby("SK_ID_CURR")[pos_numeric_prev_cols].agg(["median", "max"]))

In [ ]:
pos_client_num.columns = [col + "_" + func for col, func in pos_client_num.columns]

In [ ]:
pos_features_client = pd.concat([pos_client_counts, pos_client_special, pos_client_num, pos_window_features], axis=1)
pos_features_client.head()

In [ ]:
del POS_CASH_bal_work

del pos_num_prev
del pos_cat_prev
del pos_special_prev
del pos_features_prev
del pos_window_features

del pos_with_client
del pos_mapped
del pos_unmapped

del pos_client_counts
del pos_client_special
del pos_client_num

gc.collect()

In [ ]:
pos_features_client = (pos_features_client.astype("float32"))

gc.collect()

## Installments_payments_df

In [ ]:
installments_payments_df = pd.read_csv(path_to_data+"installments_payments.csv")

In [ ]:
installments_payments_df.info(show_counts=True)
installments_payments_df.shape

In [ ]:
home_credit_columns_description_df = pd.read_csv(path_to_data+"HomeCredit_columns_description.csv", encoding="cp1252")
inst_payment_description = home_credit_columns_description_df[home_credit_columns_description_df["Table"].astype(str).str.contains("installments_payments.csv", case = False, na=False)]
display(inst_payment_description[["Row", "Description"]])

In [ ]:
categorical_cols_inst_payment_df = []
numerical_cols_inst_payment_df = ["AMT_INSTALMENT", "AMT_PAYMENT"]
special_cols_inst_payment_df = ["SK_ID_PREV", "SK_ID_CURR", "NUM_INSTALMENT_VERSION", "NUM_INSTALMENT_NUMBER", "DAYS_INSTALMENT", "DAYS_ENTRY_PAYMENT"]

In [ ]:
print("SPECIAL:", special_cols_inst_payment_df)
print("\nCATEGORICAL:", categorical_cols_inst_payment_df)
print("\nNUMERICAL:", numerical_cols_inst_payment_df)

In [ ]:
inst_pay_work = installments_payments_df
del installments_payments_df
gc.collect()

In [ ]:
inst_pay_work["PAYMENT_DELAY"] = (inst_pay_work["DAYS_ENTRY_PAYMENT"] - inst_pay_work["DAYS_INSTALMENT"])

In [ ]:
inst_pay_work["LATE_DAYS"] = (inst_pay_work["PAYMENT_DELAY"].clip(lower=0))
inst_pay_work["EARLY_DAYS"] = ((-inst_pay_work["PAYMENT_DELAY"]).clip(lower=0))

In [ ]:
inst_pay_work["IS_PAYMENT_MISSING"] = (inst_pay_work["DAYS_ENTRY_PAYMENT"].isna().astype(int))
inst_pay_work["IS_LATE_PAYMENT"] = np.where(inst_pay_work["DAYS_ENTRY_PAYMENT"].isna(), np.nan,(inst_pay_work["PAYMENT_DELAY"] > 0).astype(int))

In [ ]:
inst_pay_work["PAYMENT_DIFF"] = (inst_pay_work["AMT_INSTALMENT"] - inst_pay_work["AMT_PAYMENT"])

In [ ]:
inst_pay_work["IS_UNDERPAID"] = np.where(inst_pay_work["AMT_PAYMENT"].isna(), np.nan, (inst_pay_work["PAYMENT_DIFF"] > 0).astype(int))

In [ ]:
inst_pay_work["PAYMENT_RATIO"] = (inst_pay_work["AMT_PAYMENT"] / inst_pay_work["AMT_INSTALMENT"].replace(0, np.nan))

In [ ]:
inst_num_prev = agg_numerical_features(inst_pay_work, "SK_ID_PREV", numerical_cols_inst_payment_df, "INST")
inst_num_prev.head()

### Special

In [ ]:
inst_special_prev = (
    inst_pay_work
    .groupby("SK_ID_PREV")
    .agg(

        INST_PAYMENT_RECORD_COUNT=(
            "NUM_INSTALMENT_NUMBER",
            "count"
        ),

        INST_PAYMENT_MISSING_COUNT=(
            "IS_PAYMENT_MISSING",
            "sum"
        ),

        INST_LATE_PAYMENT_KNOWN_COUNT=(
            "IS_LATE_PAYMENT",
            "count"
        ),

        INST_PAYMENT_AMOUNT_KNOWN_COUNT=(
            "IS_UNDERPAID",
            "count"
        ),

        INST_INSTALMENT_COUNT=(
            "NUM_INSTALMENT_NUMBER",
            "nunique"
        ),

        INST_MAX_INSTALMENT_NUMBER=(
            "NUM_INSTALMENT_NUMBER",
            "max"
        ),

        INST_MAX_VERSION=(
            "NUM_INSTALMENT_VERSION",
            "max"
        ),

        INST_DELAY_MEDIAN=(
            "PAYMENT_DELAY",
            "median"
        ),

        INST_DELAY_MAX=(
            "PAYMENT_DELAY",
            "max"
        ),

        INST_LATE_DAYS_MAX=(
            "LATE_DAYS",
            "max"
        ),

        INST_LATE_PAYMENT_COUNT=(
            "IS_LATE_PAYMENT",
            "sum"
        ),

        INST_LATE_PAYMENT_SHARE=(
            "IS_LATE_PAYMENT",
            "mean"
        ),

        INST_PAYMENT_DIFF_MEDIAN=(
            "PAYMENT_DIFF",
            "median"
        ),

        INST_PAYMENT_DIFF_MAX=(
            "PAYMENT_DIFF",
            "max"
        ),

        INST_UNDERPAID_COUNT=(
            "IS_UNDERPAID",
            "sum"
        ),

        INST_UNDERPAID_SHARE=(
            "IS_UNDERPAID",
            "mean"
        ),

        INST_PAYMENT_RATIO_MEDIAN=(
            "PAYMENT_RATIO",
            "median"
        ),

        INST_PAYMENT_RATIO_MIN=(
            "PAYMENT_RATIO",
            "min"
        ),

        INST_REQUIRED_SUM=(
            "AMT_INSTALMENT",
            "sum"
        ),

        INST_PAID_SUM=(
            "AMT_PAYMENT",
            "sum"
        )
    )
)

inst_special_prev.head()

In [ ]:
inst_special_prev["INST_TOTAL_PAYMENT_RATIO"] = (inst_special_prev["INST_PAID_SUM"] / inst_special_prev["INST_REQUIRED_SUM"].replace(0, np.nan))

In [ ]:
inst_features_prev = pd.concat([inst_num_prev, inst_special_prev], axis=1)
inst_features_prev.head()

In [ ]:
inst_with_client = (inst_features_prev.reset_index().merge(prev_id_map, on="SK_ID_PREV", how="left", validate="1:1"))
inst_with_client.head()

In [ ]:
inst_mapped = (inst_with_client[inst_with_client["SK_ID_CURR"].notna()].copy())
inst_unmapped = (inst_with_client[inst_with_client["SK_ID_CURR"].isna()].copy())

In [ ]:
inst_mapped["SK_ID_CURR"] = (inst_mapped["SK_ID_CURR"].astype("int64"))

In [ ]:
inst_sum_cols = [col for col in inst_mapped.columns if col.endswith("_COUNT") or col.endswith("_SUM")]
inst_sum_cols

In [ ]:
inst_client_sums = (inst_mapped.groupby("SK_ID_CURR")[inst_sum_cols].sum())
inst_client_sums.head()

In [ ]:
inst_client_sums["INST_LATE_PAYMENT_SHARE"] = (inst_client_sums["INST_LATE_PAYMENT_COUNT"] / inst_client_sums["INST_LATE_PAYMENT_KNOWN_COUNT"].replace(0, np.nan))

In [ ]:
inst_client_sums["INST_UNDERPAID_SHARE"] = (inst_client_sums["INST_UNDERPAID_COUNT"] / inst_client_sums["INST_PAYMENT_AMOUNT_KNOWN_COUNT"].replace(0, np.nan))

In [ ]:
inst_client_sums["INST_PAYMENT_MISSING_SHARE"] = (inst_client_sums["INST_PAYMENT_MISSING_COUNT"] / inst_client_sums[ "INST_PAYMENT_RECORD_COUNT"].replace(0, np.nan))

In [ ]:
inst_client_sums["INST_TOTAL_PAYMENT_RATIO"] = (inst_client_sums["INST_PAID_SUM"] / inst_client_sums["INST_REQUIRED_SUM"].replace(0, np.nan))

In [ ]:
inst_client_special = (
    inst_mapped
    .groupby("SK_ID_CURR")
    .agg(
        INST_CONTRACT_WITH_HISTORY_COUNT=(
            "SK_ID_PREV",
            "nunique"
        ),

        INST_DELAY_MEDIAN_CLIENT=(
            "INST_DELAY_MEDIAN",
            "median"
        ),

        INST_DELAY_MAX_CLIENT=(
            "INST_DELAY_MAX",
            "max"
        ),

        INST_LATE_DAYS_MAX_CLIENT=(
            "INST_LATE_DAYS_MAX",
            "max"
        ),

        INST_PAYMENT_DIFF_MEDIAN_CLIENT=(
            "INST_PAYMENT_DIFF_MEDIAN",
            "median"
        ),

        INST_PAYMENT_DIFF_MAX_CLIENT=(
            "INST_PAYMENT_DIFF_MAX",
            "max"
        ),

        INST_PAYMENT_RATIO_MEDIAN_CLIENT=(
            "INST_PAYMENT_RATIO_MEDIAN",
            "median"
        ),

        INST_PAYMENT_RATIO_MIN_CLIENT=(
            "INST_PAYMENT_RATIO_MIN",
            "min"
        ),

        INST_MAX_INSTALMENT_NUMBER_CLIENT=(
            "INST_MAX_INSTALMENT_NUMBER",
            "max"
        ),

        INST_MAX_VERSION_CLIENT=(
            "INST_MAX_VERSION",
            "max"
        )
    )
)

inst_client_special.head()

In [ ]:
inst_num_prev_cols = list(inst_num_prev.columns)

In [ ]:
inst_client_num = (inst_mapped.groupby("SK_ID_CURR")[inst_num_prev_cols].agg(["median", "max"]))

In [ ]:
inst_client_num.columns = [col + "_" + func for col, func in inst_client_num.columns]

In [ ]:
inst_features_client = pd.concat([inst_client_sums, inst_client_special, inst_client_num], axis=1)
inst_features_client.head()

In [ ]:
inst_pay_work["UNDERPAYMENT_AMT"] = (inst_pay_work["PAYMENT_DIFF"].clip(lower=0))

In [ ]:
def inst_window_feat(df, days, prefix):

    mask = ((df["DAYS_INSTALMENT"] >= -days) & (df["DAYS_INSTALMENT"] <= 0))

    part = df.loc[mask,
        [
            "SK_ID_CURR",
            "NUM_INSTALMENT_NUMBER",
            "IS_PAYMENT_MISSING",
            "IS_LATE_PAYMENT",
            "LATE_DAYS",
            "IS_UNDERPAID",
            "UNDERPAYMENT_AMT",
            "PAYMENT_RATIO",
            "AMT_INSTALMENT",
            "AMT_PAYMENT"
        ]
    ]

    features = (
        part
        .groupby("SK_ID_CURR")
        .agg(

            RECORD_COUNT=(
                "NUM_INSTALMENT_NUMBER",
                "count"
            ),

            PAYMENT_MISSING_COUNT=(
                "IS_PAYMENT_MISSING",
                "sum"
            ),

            LATE_KNOWN_COUNT=(
                "IS_LATE_PAYMENT",
                "count"
            ),

            LATE_COUNT=(
                "IS_LATE_PAYMENT",
                "sum"
            ),

            LATE_DAYS_MEAN=(
                "LATE_DAYS",
                "mean"
            ),

            LATE_DAYS_MAX=(
                "LATE_DAYS",
                "max"
            ),

            UNDERPAID_KNOWN_COUNT=(
                "IS_UNDERPAID",
                "count"
            ),

            UNDERPAID_COUNT=(
                "IS_UNDERPAID",
                "sum"
            ),

            UNDERPAYMENT_MEAN=(
                "UNDERPAYMENT_AMT",
                "mean"
            ),

            UNDERPAYMENT_MAX=(
                "UNDERPAYMENT_AMT",
                "max"
            ),

            PAYMENT_RATIO_MEDIAN=(
                "PAYMENT_RATIO",
                "median"
            ),

            PAYMENT_RATIO_MIN=(
                "PAYMENT_RATIO",
                "min"
            ),

            REQUIRED_SUM=(
                "AMT_INSTALMENT",
                "sum"
            ),

            PAID_SUM=(
                "AMT_PAYMENT",
                "sum"
            )
        )
    )

    features["PAYMENT_MISSING_SHARE"] = (features["PAYMENT_MISSING_COUNT"] / features["RECORD_COUNT"].replace(0, np.nan))
    features["LATE_SHARE"] = (features["LATE_COUNT"] / features["LATE_KNOWN_COUNT"].replace(0, np.nan))
    features["UNDERPAID_SHARE"] = (features["UNDERPAID_COUNT"] / features["UNDERPAID_KNOWN_COUNT"].replace(0, np.nan))
    features["TOTAL_PAYMENT_RATIO"] = (features["PAID_SUM"] / features["REQUIRED_SUM"].replace(0, np.nan))
    features = features.add_prefix(prefix + "_")
    features = features.astype("float32")
    return features

In [ ]:
inst_window_features = pd.concat([
    inst_window_feat(
        df=inst_pay_work,
        days=90,
        prefix="INST_LAST_90D"
    ),
    inst_window_feat(
        df=inst_pay_work,
        days=180,
        prefix="INST_LAST_180D"
    ),
    inst_window_feat(
        df=inst_pay_work,
        days=365,
        prefix="INST_LAST_365D"
    ),
    inst_window_feat(
        df=inst_pay_work,
        days=730,
        prefix="INST_LAST_730D"
    )
], axis=1)

inst_window_features.head()

In [ ]:
inst_features_client = pd.concat([inst_features_client, inst_window_features], axis=1)
inst_features_client.head()

In [ ]:
del inst_pay_work

del inst_num_prev
del inst_special_prev
del inst_features_prev
del inst_window_features

del inst_with_client
del inst_mapped
del inst_unmapped

del inst_client_sums
del inst_client_special
del inst_client_num

gc.collect()

In [ ]:
inst_features_client = (inst_features_client.astype("float32"))

gc.collect()

## Credit_card_balance_df

In [ ]:
credit_card_balance_df = pd.read_csv(path_to_data+"credit_card_balance.csv")

In [ ]:
credit_card_balance_df.info(show_counts=True)
credit_card_balance_df.shape

In [ ]:
home_credit_columns_description_df = pd.read_csv(path_to_data+"HomeCredit_columns_description.csv", encoding="cp1252")
credit_card_balance_description = home_credit_columns_description_df[home_credit_columns_description_df["Table"].astype(str).str.contains("credit_card_balance.csv", case = False, na=False)]
display(credit_card_balance_description[["Row", "Description"]])

In [ ]:
categorical_cols_credit_card_balance_df = ["NAME_CONTRACT_STATUS"]
numerical_cols_credit_card_balance_df = ["AMT_BALANCE", "AMT_CREDIT_LIMIT_ACTUAL", "AMT_DRAWINGS_ATM_CURRENT", "AMT_DRAWINGS_CURRENT", "AMT_DRAWINGS_OTHER_CURRENT", "AMT_DRAWINGS_POS_CURRENT", "AMT_INST_MIN_REGULARITY", "AMT_PAYMENT_CURRENT", "AMT_PAYMENT_TOTAL_CURRENT", "AMT_RECEIVABLE_PRINCIPAL", "AMT_RECIVABLE", "AMT_TOTAL_RECEIVABLE", "CNT_DRAWINGS_ATM_CURRENT", "CNT_DRAWINGS_CURRENT", "CNT_DRAWINGS_OTHER_CURRENT", "CNT_DRAWINGS_POS_CURRENT", "CNT_INSTALMENT_MATURE_CUM"]
special_cols_credit_card_balance_df = ["SK_ID_PREV", "SK_ID_CURR", "MONTHS_BALANCE", "SK_DPD", "SK_DPD_DEF"]

In [ ]:
print("SPECIAL:", special_cols_credit_card_balance_df)
print("\nCATEGORICAL:", categorical_cols_credit_card_balance_df)
print("\nNUMERICAL:", numerical_cols_credit_card_balance_df)

### Numerical

In [ ]:
credit_card_balance_work = credit_card_balance_df
del credit_card_balance_df
gc.collect()

In [ ]:
credit_card_balance_work["CC_IS_OVERDUE"] = (credit_card_balance_work["SK_DPD"] > 0).astype(int)
credit_card_balance_work["CC_IS_OVERDUE_DEF"] = (credit_card_balance_work["SK_DPD_DEF"] > 0).astype(int)

In [ ]:
credit_card_balance_work["CC_UTILIZATION"] = (credit_card_balance_work["AMT_BALANCE"] / credit_card_balance_work["AMT_CREDIT_LIMIT_ACTUAL"].replace(0, np.nan))

In [ ]:
credit_card_balance_work["CC_PAYMENT_TO_MIN"] = (credit_card_balance_work["AMT_PAYMENT_CURRENT"] / credit_card_balance_work["AMT_INST_MIN_REGULARITY"].replace(0, np.nan))

In [ ]:
credit_card_balance_work["CC_AVG_DRAWING"] = (credit_card_balance_work["AMT_DRAWINGS_CURRENT"] / credit_card_balance_work["CNT_DRAWINGS_CURRENT"].replace(0, np.nan))

In [ ]:
cc_num_prev = agg_numerical_features(credit_card_balance_work, "SK_ID_PREV", numerical_cols_credit_card_balance_df, "CC")
cc_num_prev.head()

In [ ]:
cc_window_features = agg_time_window_numerical_features(
    df=credit_card_balance_work,
    id_col="SK_ID_CURR",
    time_col="MONTHS_BALANCE",
    numerical_cols=[
        "AMT_BALANCE",
        "AMT_CREDIT_LIMIT_ACTUAL",
        "AMT_DRAWINGS_CURRENT",
        "AMT_PAYMENT_CURRENT",
        "AMT_INST_MIN_REGULARITY",
        "SK_DPD",
        "SK_DPD_DEF",
        "CC_IS_OVERDUE",
        "CC_IS_OVERDUE_DEF",
        "CC_UTILIZATION",
        "CC_PAYMENT_TO_MIN",
        "CC_AVG_DRAWING"
    ],
    prefix="CC",
    window_sizes=(3, 6, 12),
    unit="M"
)

cc_window_features.head()

### Categorical

In [ ]:
cc_cat_prev = agg_categorical_features(credit_card_balance_work, "SK_ID_PREV", categorical_cols_credit_card_balance_df, "CC")
cc_cat_prev.head()

### Special

In [ ]:
cc_special_prev = (credit_card_balance_work.groupby("SK_ID_PREV").agg(
        CC_MONTH_COUNT=(
            "MONTHS_BALANCE",
            "count"
        ),

        CC_OLDEST_MONTH=(
            "MONTHS_BALANCE",
            "min"
        ),

        CC_MOST_RECENT_MONTH=(
            "MONTHS_BALANCE",
            "max"
        ),

        CC_MAX_DPD=(
            "SK_DPD",
            "max"
        ),

        CC_MAX_DPD_DEF=(
            "SK_DPD_DEF",
            "max"
        ),

        CC_OVERDUE_MONTH_COUNT=(
            "CC_IS_OVERDUE",
            "sum"
        ),

        CC_OVERDUE_DEF_MONTH_COUNT=(
            "CC_IS_OVERDUE_DEF",
            "sum"
        ),

        CC_UTILIZATION_MEDIAN=(
            "CC_UTILIZATION",
            "median"
        ),

        CC_UTILIZATION_MAX=(
            "CC_UTILIZATION",
            "max"
        ),

        CC_PAYMENT_TO_MIN_MEDIAN=(
            "CC_PAYMENT_TO_MIN",
            "median"
        ),

        CC_PAYMENT_TO_MIN_MAX=(
            "CC_PAYMENT_TO_MIN",
            "max"
        ),

        CC_AVG_DRAWING_MEDIAN=(
            "CC_AVG_DRAWING",
            "median"
        ),

        CC_AVG_DRAWING_MAX=(
            "CC_AVG_DRAWING",
            "max"
        ),

        CC_DRAWINGS_SUM=(
            "AMT_DRAWINGS_CURRENT",
            "sum"
        ),

        CC_PAYMENT_SUM=(
            "AMT_PAYMENT_CURRENT",
            "sum"
        )
    )
)

cc_special_prev.head()

In [ ]:
cc_special_prev["CC_HISTORY_LENGTH"] = (cc_special_prev["CC_MOST_RECENT_MONTH"] - cc_special_prev["CC_OLDEST_MONTH"])

In [ ]:
cc_features_prev = pd.concat([cc_num_prev, cc_cat_prev, cc_special_prev], axis=1)
cc_features_prev.head()

In [ ]:
cc_with_client = (cc_features_prev.reset_index().merge(prev_id_map, on="SK_ID_PREV", how="left", validate="1:1"))
cc_with_client.head()

In [ ]:
cc_mapped = (cc_with_client[cc_with_client["SK_ID_CURR"].notna()].copy())

cc_unmapped = (cc_with_client[cc_with_client["SK_ID_CURR"].isna()].copy())
cc_mapped["SK_ID_CURR"] = (cc_mapped["SK_ID_CURR"].astype("int64"))

In [ ]:
cc_sum_cols = [col for col in cc_mapped.columns if col.endswith("_COUNT") or col.endswith("_SUM")]

In [ ]:
cc_client_sums = (cc_mapped.groupby("SK_ID_CURR")[cc_sum_cols].sum())
cc_client_sums.head()

In [ ]:
cc_client_sums["CC_OVERDUE_MONTH_SHARE"] = (cc_client_sums["CC_OVERDUE_MONTH_COUNT"] / cc_client_sums["CC_MONTH_COUNT"].replace(0, np.nan))
cc_client_sums["CC_OVERDUE_DEF_MONTH_SHARE"] = (cc_client_sums["CC_OVERDUE_DEF_MONTH_COUNT"] / cc_client_sums["CC_MONTH_COUNT"].replace(0, np.nan))

In [ ]:
cc_status_count_cols = [col for col in cc_client_sums.columns if col.startswith("CC_NAME_CONTRACT_STATUS_") and col.endswith("_COUNT")]

In [ ]:
for col in cc_status_count_cols:

    share_col = col.replace("_COUNT", "_SHARE")
    cc_client_sums[share_col] = (cc_client_sums[col] / cc_client_sums["CC_MONTH_COUNT"].replace(0, np.nan))

In [ ]:
cc_client_special = (
    cc_mapped
    .groupby("SK_ID_CURR")
    .agg(
        CC_CONTRACT_WITH_HISTORY_COUNT=(
            "SK_ID_PREV",
            "nunique"
        ),

        CC_HISTORY_LENGTH_MAX=(
            "CC_HISTORY_LENGTH",
            "max"
        ),

        CC_HISTORY_LENGTH_MEDIAN=(
            "CC_HISTORY_LENGTH",
            "median"
        ),

        CC_OLDEST_MONTH_CLIENT=(
            "CC_OLDEST_MONTH",
            "min"
        ),

        CC_MOST_RECENT_MONTH_CLIENT=(
            "CC_MOST_RECENT_MONTH",
            "max"
        ),

        CC_MAX_DPD_CLIENT=(
            "CC_MAX_DPD",
            "max"
        ),

        CC_MAX_DPD_DEF_CLIENT=(
            "CC_MAX_DPD_DEF",
            "max"
        ),

        CC_UTILIZATION_MEDIAN_CLIENT=(
            "CC_UTILIZATION_MEDIAN",
            "median"
        ),

        CC_UTILIZATION_MAX_CLIENT=(
            "CC_UTILIZATION_MAX",
            "max"
        ),

        CC_PAYMENT_TO_MIN_MEDIAN_CLIENT=(
            "CC_PAYMENT_TO_MIN_MEDIAN",
            "median"
        ),

        CC_PAYMENT_TO_MIN_MAX_CLIENT=(
            "CC_PAYMENT_TO_MIN_MAX",
            "max"
        ),

        CC_AVG_DRAWING_MEDIAN_CLIENT=(
            "CC_AVG_DRAWING_MEDIAN",
            "median"
        ),

        CC_AVG_DRAWING_MAX_CLIENT=(
            "CC_AVG_DRAWING_MAX",
            "max"
        )
    )
)

cc_client_special.head()

In [ ]:
cc_num_prev_cols = list(cc_num_prev.columns)
cc_client_num = (cc_mapped.groupby("SK_ID_CURR")[cc_num_prev_cols].agg(["median", "max"]))
cc_client_num.columns = [col + "_" + func for col, func in cc_client_num.columns]

In [ ]:
cc_features_client = pd.concat([cc_client_sums, cc_client_special, cc_client_num, cc_window_features], axis=1)
cc_features_client.head()

In [ ]:
del credit_card_balance_work

del cc_num_prev
del cc_cat_prev
del cc_special_prev
del cc_features_prev
del cc_window_features

del cc_with_client
del cc_mapped
del cc_unmapped

del cc_client_sums
del cc_client_special
del cc_client_num
del prev_id_map

gc.collect()

In [ ]:
cc_features_client = (cc_features_client.astype("float32"))

gc.collect()

# Build final tables

## Add the left part

In [ ]:
bureau_branch_features = pd.concat([bureau_features, bureau_balance_features_client], axis=1)

print("Размер:", bureau_branch_features.shape)
print("SK_ID_CURR уникален:", bureau_branch_features.index.is_unique)
print("Дубликатов колонок:", bureau_branch_features.columns.duplicated().sum())

## Add the right part

In [ ]:
application_train_full = application_train_df.copy()
application_test_full = application_test_df.copy()

In [ ]:
train_rows_before = application_train_full.shape[0]
test_rows_before = application_test_full.shape[0]

print("Train:", application_train_full.shape)
print("Test:", application_test_full.shape)

## Add bureau branch

In [ ]:
application_train_full = application_train_full.merge(bureau_branch_features.reset_index(), on = "SK_ID_CURR", how="left", validate="one_to_one")
application_test_full = application_test_full.merge(bureau_branch_features.reset_index(), on = "SK_ID_CURR", how="left", validate="one_to_one")

In [ ]:
del bureau_branch_features
del bureau_features
del bureau_balance_features_client

gc.collect()

In [ ]:
application_train_full = (
    application_train_full
    .set_index("SK_ID_CURR")
)

application_test_full = (
    application_test_full
    .set_index("SK_ID_CURR")
)

## Add previous_application branch

### Add previous_application to application_train_full and to application_test_full

In [ ]:
application_train_full = application_train_full.join(prev_features, how="left", validate="one_to_one")
application_test_full = application_test_full.join(prev_features, how="left", validate="one_to_one")

In [ ]:
del prev_features
gc.collect()

### Add POS_CASH_balance_df to application_train_full and to application_test_full

In [ ]:
pos_features_client.index = (pos_features_client.index.astype("int64"))

In [ ]:
application_train_full = application_train_full.join(pos_features_client, how="left", validate="one_to_one")
application_test_full = application_test_full.join(pos_features_client, how="left", validate="one_to_one")

In [ ]:
del pos_features_client
gc.collect()

### Add installments_payments_df to application_train_full and to application_test_full

In [ ]:
inst_features_client.index = (inst_features_client.index.astype("int64"))

In [ ]:
application_train_full = application_train_full.join(inst_features_client, how="left", validate="one_to_one")
application_test_full = application_test_full.join(inst_features_client, how="left", validate="one_to_one")

In [ ]:
del inst_features_client
gc.collect()

### Add credit_card_balance_df to application_train_full and to application_test_full

In [ ]:
cc_features_client.index = (cc_features_client.index.astype("int64"))

In [ ]:
application_train_full = application_train_full.join(cc_features_client, how="left", validate="one_to_one")
application_test_full = application_test_full.join(cc_features_client, how="left", validate="one_to_one")

In [ ]:
del cc_features_client
gc.collect()

In [ ]:
application_train_full = (application_train_full.reset_index())
application_test_full = (application_test_full.reset_index())

In [ ]:
print("Train:", application_train_full.shape)
print("Test:", application_test_full.shape)

### kNN

In [ ]:
from sklearn.manifold import TSNE

In [ ]:
feat_knn = [
    "EXT_SOURCE_2",
    "EXT_SOURCE_3",
    "EXT_SOURCE_1",
    "ORGANIZATION_TYPE",
    # "BUREAU_MAX_DEBT_RATIO",
    "AMT_ANNUITY",
    "CODE_GENDER",
    "AMT_GOODS_PRICE"
]
feat_knn_df = application_train_df[feat_knn]

feat_knn_df.head()

In [ ]:
!pip install category-encoders

In [ ]:
import category_encoders

In [ ]:
enc=category_encoders.OrdinalEncoder(cols=["ORGANIZATION_TYPE", "CODE_GENDER"])
# for col in ["ORGANIZATION_TYPE", "CODE_GENDER"]:
#     feat_knn_df[col] = enc.fit_transform(feat_knn_df[col])
feat_knn_df = enc.fit_transform(feat_knn_df)
feat_knn_df

In [ ]:
for col in feat_knn_df:
    median = feat_knn_df[col].median()
    feat_knn_df.fillna(median, inplace=True)

In [ ]:
feat_knn_df

In [ ]:
tsne = TSNE(n_components=2, perplexity=30, max_iter=300)
feat_2D = tsne.fit_transform(feat_knn_df)

In [ ]:
plt.scatter(feat_2D[:, 0], feat_2D[:, 1], c=application_train_df["TARGET"], cmap='jet', alpha=0.7)
plt.colorbar()
plt.title("t-SNE Visualization")
plt.show()

# Save final tables

In [ ]:
save_path = "~/PycharmProjects/home-credit-default-risk/data/processed/"

In [ ]:
application_train_full.to_csv(save_path+"application_train.csv", index=False)

In [ ]:
application_test_full.to_csv(save_path+"application_test.csv", index=False)

In [ ]:
print("Train:", application_train_full.shape)
print("Test:", application_test_full.shape)

In [ ]:
del application_train_full
del application_test_full

gc.collect()